# FMRec — Stateful Retrieval + LLM Ranking

Notebook này gộp hai bước:

1. **Retrieve failure lessons** bằng LightGCN user embeddings.
2. **Rank candidates** bằng local LLM (Vanilla hoặc FMRec).

Mục tiêu là tránh load lại các thành phần nặng giữa hai script:

- `RecDataset`, metadata, instructions: **load 1 lần**
- frozen/random negative candidates: **load/tạo 1 lần**
- LightGCN embeddings: **load 1 lần**
- failure memory: **load 1 lần**
- local LLM: **load 1 lần**
- có thể thay `TOP_K_NEIGHBORS`, retrieve lại, rồi rank lại mà **không reload LLM**

> Notebook nên được mở từ thư mục gốc của MemRec hoặc một thư mục con của repo. Code bên dưới tự tìm `src/`.

In [ ]:
# ============================================================
# 0. CONFIG — chỉnh chủ yếu ở cell này
# ============================================================

DATA_PATH = "data/processed/instructrec-books/instructrec-books.inter"
EVAL_USER_LIST = "data/eval_user_samples/eval_user_sample_10_instructrec-books.json"

# Frozen MemRec-test candidates. Nếu file đã có thì chỉ load lại.
CANDIDATE_FILE = "results/books_candidates_1k_memrec_test.json"

LIGHTGCN_EMBEDDINGS = "results/lightgcn_books/lightgcn_embeddings.json"
FAILURE_MEMORY = "results/fmrec_lessons_books_r2/failure_memory.jsonl"

RETRIEVAL_OUTPUT_DIR = "results/fmrec_retrieval_books"
FMREC_OUTPUT_DIR = "results/llmrank_fmrec_history"
VANILLA_OUTPUT_DIR = "results/llmrank_vanilla_history"

MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

SEED = 42
MAX_USERS = None
N_CANDIDATES = 10

TOP_K_NEIGHBORS = 3
INCLUDE_SELF = True
LABEL_SOURCES = True

# IMPORTANT:
# Script llmrank_fmrec_history hiện tại đặt default=3.
# Vì retrieval có thể trả 1 personal + 3 collaborative = 4,
# MAX_LESSON_FACTS=3 giữ hành vi EXACT của script ranking hiện tại.
# Đặt =4 nếu muốn thực sự đưa đủ tối đa 4 lessons vào prompt.
MAX_LESSON_FACTS = 4

USE_INSTRUCTION = True
USE_DESCRIPTION = False

BATCH_SIZE = 8
MAX_NEW_TOKENS = 1000
MAX_RETRIES = 1

In [ ]:
# ============================================================
# 1. IMPORTS + FIND MEMREC REPO
# ============================================================

import html
import json
import math
import os
import random
import re
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
from tqdm.auto import tqdm


def find_repo_root(start: Optional[Path] = None) -> Path:
    """Find nearest directory containing src/."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "src").is_dir():
            return p
    raise RuntimeError(
        f"Cannot find MemRec repo root from {start}. "
        "Open this notebook inside the MemRec repository."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import RecDataset

print("MemRec repo:", REPO_ROOT)

In [ ]:
# ============================================================
# 2. SHARED HELPERS: seed, metadata, users, candidates
# ============================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clean_text(value: Any, max_chars: int = 0) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        value = " ".join(str(x) for x in value if str(x).strip())
    elif isinstance(value, dict):
        value = " ".join(str(x) for x in value.values() if str(x).strip())

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if max_chars > 0 and len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."
    return text


def get_candidate_item(
    dataset: RecDataset,
    item_id: int,
    use_description: bool = False,
) -> Dict[str, Any]:
    meta = (dataset.item_metadata or {}).get(int(item_id), {})

    tags = meta.get("tags", [])
    if tags is None:
        tags = []
    if isinstance(tags, str):
        tags = [tags]
    elif not isinstance(tags, list):
        tags = list(tags) if isinstance(tags, (tuple, set)) else []

    row = {
        "id": int(item_id),
        "title": meta.get("title", "N/A"),
        "tags": [str(x) for x in tags],
    }

    if use_description:
        row["description"] = clean_text(meta.get("description", ""), 200)

    return row


def get_history_items(dataset: RecDataset, uid: int) -> List[Dict[str, Any]]:
    """Full observed TRAIN history, oldest -> newest."""
    rows = []
    for item_id in dataset.get_user_train_items(uid):
        item_id = int(item_id)
        meta = (dataset.item_metadata or {}).get(item_id, {})
        title = clean_text(meta.get("title", ""), 300) or f"Item {item_id}"
        rows.append({"id": item_id, "title": title})
    return rows


def load_eval_users(
    path: Optional[str],
    dataset: RecDataset,
    max_users: Optional[int],
    seed: int,
) -> List[int]:
    if path:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        ids = obj.get("user_ids") if isinstance(obj, dict) else obj
        if ids is None:
            raise ValueError(f"No user_ids found in {path}")

        users = [int(x) for x in ids]
        users = [u for u in users if u in dataset.test_data]

        if max_users is not None and max_users > 0:
            users = users[:max_users]
        return users

    all_users = list(dataset.test_data.keys())

    if max_users is not None and 0 < max_users < len(all_users):
        random.seed(seed)
        return random.sample(all_users, max_users)

    return all_users


def make_memrec_test_candidates(
    dataset: RecDataset,
    users: List[int],
    n_candidates: int,
) -> Dict[int, Dict[str, Any]]:
    out = {}

    for uid in users:
        target = int(dataset.test_data[uid])

        all_items = set(range(dataset.n_items))
        train_history = set(dataset.get_user_train_items(uid))
        negative_pool = list(all_items - train_history - {target})

        if len(negative_pool) < n_candidates - 1:
            raise ValueError(
                f"User {uid}: only {len(negative_pool)} negatives available; "
                f"need {n_candidates - 1}"
            )

        negative_items = random.sample(negative_pool, n_candidates - 1)
        candidate_ids = [target] + [int(x) for x in negative_items]
        random.shuffle(candidate_ids)

        out[uid] = {
            "target": target,
            "candidates": candidate_ids,
        }

    return out


def load_or_make_candidates(
    dataset: RecDataset,
    users: List[int],
    n_candidates: int,
    seed: int,
    path: Optional[str],
) -> Dict[int, Dict[str, Any]]:
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict) and "candidate_source" in obj:
            source = obj.get("candidate_source")
            if source != "memrec_test":
                raise ValueError(
                    f"{path} has candidate_source={source!r}; "
                    "expected 'memrec_test'."
                )
        elif isinstance(obj, dict) and "users" in obj:
            raise ValueError(
                f"{path} has no candidate_source marker. "
                "Delete/rename it if it is an old candidate file."
            )

        raw = obj.get("users", obj)
        out = {
            int(uid): {
                "target": int(row["target"]),
                "candidates": [int(x) for x in row["candidates"]],
            }
            for uid, row in raw.items()
        }

        for uid in users:
            if uid not in out:
                raise KeyError(f"Candidate file missing user {uid}")
            if out[uid]["target"] != int(dataset.test_data[uid]):
                raise ValueError(f"Target mismatch for user {uid}")
            if len(out[uid]["candidates"]) != n_candidates:
                raise ValueError(f"Candidate count mismatch for user {uid}")

        print(f"✓ Loaded frozen MemRec-test candidates: {path}")
        return out

    out = make_memrec_test_candidates(dataset, users, n_candidates)

    if path:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "candidate_source": "memrec_test",
                    "seed": seed,
                    "n_candidates": n_candidates,
                    "users": {str(uid): row for uid, row in out.items()},
                },
                f,
                indent=2,
            )
        print(f"✓ Saved frozen MemRec-test candidates: {path}")

    return out

## Load dataset + candidates **một lần**

Sau cell này, `dataset`, `users`, `candidates` nằm trong RAM. Nếu chỉ đổi retrieval hoặc rerank, **không chạy lại cell này**.

In [ ]:
# ============================================================
# 3. LOAD DATASET + USERS + CANDIDATES ONCE
# ============================================================

seed_all(SEED)

dataset = RecDataset(DATA_PATH, seed=SEED)
dataset.load_item_metadata()
dataset.load_instructions()

users = load_eval_users(
    EVAL_USER_LIST,
    dataset,
    MAX_USERS,
    SEED,
)

candidates = load_or_make_candidates(
    dataset,
    users,
    N_CANDIDATES,
    SEED,
    CANDIDATE_FILE,
)

# Precompute prompt-side inputs once as well.
# These are reused across FMRec/Vanilla and across different retrieval K.
history_cache = {
    uid: get_history_items(dataset, uid)
    for uid in tqdm(users, desc="Precompute histories")
}

candidate_items_cache = {
    uid: [
        get_candidate_item(
            dataset,
            item_id,
            use_description=USE_DESCRIPTION,
        )
        for item_id in candidates[uid]["candidates"]
    ]
    for uid in tqdm(users, desc="Precompute candidate metadata")
}

instruction_cache = {}
for uid in users:
    instruction = ""
    if (
        USE_INSTRUCTION
        and dataset.instructions
        and uid in dataset.instructions
    ):
        instruction = str(
            dataset.instructions[uid].get("instruction", "") or ""
        )
    instruction_cache[uid] = instruction

print(f"✓ Dataset + prompt-side caches ready")
print(f"  users      : {len(users)}")
print(f"  n_items    : {dataset.n_items}")
print(f"  candidates : {N_CANDIDATES}/user")
print(f"  histories  : {len(history_cache)} cached")
print(f"  cand meta  : {len(candidate_items_cache)} users cached")

In [ ]:
# ============================================================
# 4. RETRIEVAL HELPERS
# ============================================================

def load_user_embeddings(path: str) -> Dict[int, np.ndarray]:
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    raw_users = payload.get("users", {})
    if not raw_users:
        raise ValueError(f"No user embeddings found in {path}")

    out = {}
    for uid, vec in raw_users.items():
        arr = np.asarray(vec, dtype=np.float32)
        norm = float(np.linalg.norm(arr))
        if norm <= 1e-12:
            continue
        out[int(uid)] = arr / norm

    print(f"✓ Loaded LightGCN embeddings for {len(out)} users")
    return out


def load_failure_memory(path: str) -> Dict[int, List[Dict[str, Any]]]:
    by_user = defaultdict(list)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)
            source_uid = int(row["source_user_id"])
            lesson = str(row.get("lesson", "") or "").strip()

            if not lesson or lesson.upper() == "NONE":
                continue

            row["source_user_id"] = source_uid
            row["lesson"] = lesson
            row["confidence"] = float(row.get("confidence", 0.0) or 0.0)
            by_user[source_uid].append(row)

    for uid in by_user:
        by_user[uid].sort(
            key=lambda x: (
                -float(x.get("confidence", 0.0)),
                str(x.get("memory_id", "")),
            )
        )

    n_lessons = sum(len(v) for v in by_user.values())
    print(
        f"✓ Loaded {n_lessons} usable lessons "
        f"from {len(by_user)} source users"
    )
    return dict(by_user)


def best_lesson_for_user(
    source_uid: int,
    lessons_by_user: Dict[int, List[Dict[str, Any]]],
) -> Optional[Dict[str, Any]]:
    rows = lessons_by_user.get(source_uid, [])
    return rows[0] if rows else None


def retrieve_neighbor_users(
    query_uid: int,
    embeddings: Dict[int, np.ndarray],
    lessons_by_user: Dict[int, List[Dict[str, Any]]],
    top_k: int,
) -> List[Dict[str, Any]]:
    query_vec = embeddings.get(query_uid)
    if query_vec is None:
        return []

    candidates_local = []

    for source_uid, source_lessons in lessons_by_user.items():
        if source_uid == query_uid or not source_lessons:
            continue

        source_vec = embeddings.get(source_uid)
        if source_vec is None:
            continue

        candidates_local.append({
            "source_user_id": source_uid,
            "similarity": float(np.dot(query_vec, source_vec)),
        })

    candidates_local.sort(
        key=lambda x: (-x["similarity"], x["source_user_id"])
    )
    return candidates_local[:top_k]


def retrieve_for_user(
    uid: int,
    embeddings: Dict[int, np.ndarray],
    lessons_by_user: Dict[int, List[Dict[str, Any]]],
    top_k_neighbors: int,
    include_self: bool,
    label_sources: bool,
) -> Dict[str, Any]:
    selected = []

    if include_self:
        own = best_lesson_for_user(uid, lessons_by_user)
        if own is not None:
            selected.append({
                "source_type": "personal",
                "source_user_id": uid,
                "similarity": 1.0,
                "memory_id": own.get("memory_id", ""),
                "confidence": float(own.get("confidence", 0.0)),
                "lesson": own["lesson"],
            })

    neighbors = retrieve_neighbor_users(
        uid,
        embeddings,
        lessons_by_user,
        top_k_neighbors,
    )

    for neighbor in neighbors:
        source_uid = int(neighbor["source_user_id"])
        memory = best_lesson_for_user(source_uid, lessons_by_user)
        if memory is None:
            continue

        selected.append({
            "source_type": "collaborative",
            "source_user_id": source_uid,
            "similarity": float(neighbor["similarity"]),
            "memory_id": memory.get("memory_id", ""),
            "confidence": float(memory.get("confidence", 0.0)),
            "lesson": memory["lesson"],
        })

    memory_facts = []
    for row in selected:
        lesson = row["lesson"]
        if label_sources:
            prefix = (
                "Personal failure lesson: "
                if row["source_type"] == "personal"
                else "Collaborative failure lesson: "
            )
            lesson = prefix + lesson
        memory_facts.append(lesson)

    return {
        "user_id": uid,
        "memory_facts": memory_facts,
        "selected": selected,
    }


def retrieve_all(
    users: List[int],
    embeddings: Dict[int, np.ndarray],
    lessons_by_user: Dict[int, List[Dict[str, Any]]],
    top_k_neighbors: int = 3,
    include_self: bool = True,
    label_sources: bool = True,
):
    retrieved_lessons = {}
    audit_rows = []

    n_with_self = 0
    n_with_collab = 0
    n_with_any = 0
    total_collab = 0
    total_lessons = 0
    missing_embedding = 0

    for uid in tqdm(users, desc="Retrieve FMRec lessons"):
        result = retrieve_for_user(
            uid,
            embeddings,
            lessons_by_user,
            top_k_neighbors,
            include_self,
            label_sources,
        )

        selected = result["selected"]
        personal = [x for x in selected if x["source_type"] == "personal"]
        collaborative = [
            x for x in selected if x["source_type"] == "collaborative"
        ]

        retrieved_lessons[uid] = result["memory_facts"]

        audit_rows.append({
            "user_id": uid,
            "n_personal": len(personal),
            "n_collaborative": len(collaborative),
            "selected_lessons": selected,
        })

        missing_embedding += int(uid not in embeddings)
        n_with_self += int(bool(personal))
        n_with_collab += int(bool(collaborative))
        n_with_any += int(bool(selected))
        total_collab += len(collaborative)
        total_lessons += len(selected)

    n = len(users)
    summary = {
        "method": "FMRec LightGCN lesson retrieval",
        "n_eval_users": n,
        "lightgcn_embeddings": LIGHTGCN_EMBEDDINGS,
        "failure_memory": FAILURE_MEMORY,
        "top_k_neighbors": top_k_neighbors,
        "include_self": include_self,
        "label_sources": label_sources,
        "max_lessons_per_user": (1 if include_self else 0) + top_k_neighbors,
        "users_with_personal_lesson": n_with_self,
        "personal_coverage": n_with_self / n if n else 0.0,
        "users_with_collaborative_lesson": n_with_collab,
        "collaborative_coverage": n_with_collab / n if n else 0.0,
        "users_with_any_lesson": n_with_any,
        "memory_coverage": n_with_any / n if n else 0.0,
        "avg_collaborative_lessons": total_collab / n if n else 0.0,
        "avg_total_lessons": total_lessons / n if n else 0.0,
        "users_missing_lightgcn_embedding": missing_embedding,
    }

    return retrieved_lessons, audit_rows, summary


def save_retrieval(
    retrieved_lessons: Dict[int, List[str]],
    audit_rows: List[Dict[str, Any]],
    summary: Dict[str, Any],
    output_dir: str,
):
    os.makedirs(output_dir, exist_ok=True)

    lessons_path = os.path.join(output_dir, "retrieved_lessons_by_user.jsonl")
    audit_path = os.path.join(output_dir, "retrieval_audit.jsonl")
    summary_path = os.path.join(output_dir, "summary.json")

    with open(lessons_path, "w", encoding="utf-8") as f:
        for uid in users:
            f.write(json.dumps({
                "user_id": uid,
                "memory_facts": retrieved_lessons.get(uid, []),
            }, ensure_ascii=False) + "\n")

    with open(audit_path, "w", encoding="utf-8") as f:
        for row in audit_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    summary_to_save = dict(summary)
    summary_to_save["outputs"] = {
        "llmrank_lessons": lessons_path,
        "retrieval_audit": audit_path,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_to_save, f, indent=2, ensure_ascii=False)

    print("✓ Saved retrieval outputs:", output_dir)

## Load LightGCN + failure memory **một lần**

Nếu chỉ đổi `TOP_K_NEIGHBORS`, `INCLUDE_SELF`, hoặc `LABEL_SOURCES`, giữ nguyên cell load này và chỉ chạy lại cell retrieval phía dưới.

In [ ]:
# ============================================================
# 5. LOAD RETRIEVAL STATE ONCE
# ============================================================

user_embeddings = load_user_embeddings(LIGHTGCN_EMBEDDINGS)
failure_memory_by_user = load_failure_memory(FAILURE_MEMORY)

In [ ]:
# ============================================================
# 6. RUN / RE-RUN RETRIEVAL IN MEMORY
# ============================================================
# Cell này rất rẻ so với load LLM. Có thể đổi K rồi chạy lại.

retrieved_lessons, retrieval_audit, retrieval_summary = retrieve_all(
    users=users,
    embeddings=user_embeddings,
    lessons_by_user=failure_memory_by_user,
    top_k_neighbors=TOP_K_NEIGHBORS,
    include_self=INCLUDE_SELF,
    label_sources=LABEL_SOURCES,
)

print(json.dumps(retrieval_summary, indent=2, ensure_ascii=False))

# Giữ output tương thích script retrieve_fmrec_lessons.py
save_retrieval(
    retrieved_lessons,
    retrieval_audit,
    retrieval_summary,
    RETRIEVAL_OUTPUT_DIR,
)

In [ ]:
# ============================================================
# 7. PROMPT + PARSING + METRICS
# ============================================================

def build_memrec_vanilla_prompt(
    user_id: int,
    instruction: Optional[str],
    history_items: List[Dict[str, Any]],
    candidates: List[Dict[str, Any]],
    failure_lessons: Optional[List[str]] = None,
) -> str:
    parts = [
        "You are an intelligent recommendation scoring system. "
        "Evaluate how well each candidate matches the target user's preferences.",
        f"\n\n**Target User:** User {user_id}",
    ]

    if instruction:
        parts.append(f"\n\n**User Instruction:**\n{instruction}")
    else:
        parts.append("\n\n**User Instruction:**\nNo specific instruction provided.")

    parts.append("\n\n**Observed Training History (oldest to newest):**")
    if history_items:
        for idx, item in enumerate(history_items, 1):
            parts.append(f"\n  {idx}. {item['title']}")
    else:
        parts.append("\n  No observed training history.")

    if failure_lessons:
        parts.append("\n\n**Retrieved Failure Lessons:**")
        for idx, fact in enumerate(failure_lessons, 1):
            parts.append(f"\n  {idx}. {fact}")
        parts.append(
            "\nUse these lessons only when relevant to the current user's "
            "instruction, history, and candidate items."
        )

    parts.append("\n\n**Candidate Items:**")
    for c in candidates:
        cid = int(c["id"])
        title = c.get("title", f"Item {cid}")
        description = c.get("description", "")
        tags = c.get("tags", [])

        if description:
            if len(description) > 200:
                description = description[:200] + "..."
            parts.append(f"\n  • Item {cid}: {title}. {description}")
        elif tags:
            tags_str = ", ".join(str(x) for x in tags[:5])
            parts.append(f"\n  • Item {cid}: {title} (Tags: {tags_str})")
        else:
            parts.append(f"\n  • Item {cid}: {title}")

    parts.append("""

**Your Task:**
For every candidate item above, provide a relevance score from 0 to 1:
  • 1.0 = excellent match
  • 0.5 = moderate match
  • 0.0 = poor match

Base the ranking on the user's instruction, observed training history, and
candidate characteristics. If failure lessons are provided, use them only as
additional evidence.

Return a JSON object with exactly one score row for every candidate:
{
  "scores": [
    {
      "item_id": 123,
      "score": 0.8,
      "rationale": "brief reason"
    }
  ]
}
""")
    return "".join(parts)


def extract_json(text: str) -> Dict[str, Any]:
    text = str(text or "").strip()

    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0]
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0]

    start = text.find("{")
    end = text.rfind("}") + 1

    if start >= 0 and end > start:
        text = text[start:end]

    return json.loads(text)


def recover_score_rows(text: str) -> List[Dict[str, Any]]:
    pattern = re.compile(
        r'"item_id"\s*:\s*"?(\d+)"?'
        r'[^{}]{0,120}?'
        r'"score"\s*:\s*"?([0-9]*\.?[0-9]+)"?',
        re.I,
    )

    return [
        {
            "item_id": int(match.group(1)),
            "score": float(match.group(2)),
            "rationale": "Recovered from malformed JSON",
        }
        for match in pattern.finditer(str(text or ""))
    ]


def parse_ranking(raw: str, candidate_ids: List[int]):
    try:
        rows = extract_json(raw).get("scores", [])
    except Exception:
        rows = recover_score_rows(raw)

    candidate_set = set(int(x) for x in candidate_ids)
    original_index = {
        int(item_id): idx
        for idx, item_id in enumerate(candidate_ids)
    }

    seen = set()
    parsed = []

    for row in rows if isinstance(rows, list) else []:
        if not isinstance(row, dict):
            continue

        try:
            item_id = int(row.get("item_id"))
        except Exception:
            continue

        if item_id not in candidate_set or item_id in seen:
            continue

        try:
            score = float(row.get("score", 0.0))
        except Exception:
            score = 0.0

        parsed.append({
            "item_id": item_id,
            "score": max(0.0, min(1.0, score)),
            "rationale": str(row.get("rationale", "")),
        })
        seen.add(item_id)

    valid = len(seen) == len(candidate_ids)

    for item_id in candidate_ids:
        item_id = int(item_id)
        if item_id not in seen:
            parsed.append({
                "item_id": item_id,
                "score": -1.0,
                "rationale": "Missing from LLM output",
            })

    parsed.sort(
        key=lambda x: (-x["score"], original_index[x["item_id"]])
    )
    ranking = [row["item_id"] for row in parsed]

    return ranking, valid, parsed


def metrics(positions: List[int]) -> Dict[str, float]:
    if not positions:
        return {}

    n = len(positions)
    out = {}

    for k in (1, 3, 5, 10):
        out[f"Hit@{k}"] = sum(pos < k for pos in positions) / n
        out[f"NDCG@{k}"] = (
            sum(
                (1.0 / math.log2(pos + 2)) if pos < k else 0.0
                for pos in positions
            )
            / n
        )

    out["MRR"] = sum(1.0 / (pos + 1) for pos in positions) / n
    return out


def append_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_resume(path: str):
    done, rows = set(), []

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                done.add(int(row["user_id"]))
                rows.append(row)

    return done, rows

In [ ]:
# ============================================================
# 8. LOCAL LLM CLASS
# ============================================================

class UnslothBatchLLM:
    def __init__(
        self,
        model_name: str,
        max_seq_length: int,
        load_in_4bit: bool,
    ):
        from unsloth import FastModel

        print(f"Loading model ONCE: {model_name}")

        self.model, self.processor = FastModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,
        )
        FastModel.for_inference(self.model)

        self.tokenizer = getattr(
            self.processor,
            "tokenizer",
            self.processor,
        )

        self.max_seq_length = int(max_seq_length)

        self.tokenizer.padding_side = "left"
        self.tokenizer.truncation_side = "left"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        try:
            self.model.generation_config.pad_token_id = (
                self.tokenizer.pad_token_id
            )
        except Exception:
            pass

        self.calls = 0
        self.examples = 0
        self.seconds = 0.0

    def generate_batch(
        self,
        prompts: List[str],
        max_new_tokens: int,
    ) -> List[str]:

        batch_messages = [
            [{"role": "user", "content": prompt}]
            for prompt in prompts
        ]

        texts = []
        for messages in batch_messages:
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            texts.append(text)

        inp = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_seq_length,
        )

        inp = {
            key: value.to(self.model.device)
            for key, value in inp.items()
        }

        width = int(inp["input_ids"].shape[1])

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            out = self.model.generate(
                **inp,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        self.calls += 1
        self.examples += len(prompts)
        self.seconds += elapsed

        return [
            self.tokenizer.decode(
                row[width:],
                skip_special_tokens=True,
            ).strip()
            for row in out
        ]

    def stats(self) -> Dict[str, Any]:
        return {
            "generate_calls": self.calls,
            "examples": self.examples,
            "seconds": self.seconds,
            "examples_per_second": (
                self.examples / self.seconds if self.seconds else 0.0
            ),
        }

## Load LLM **một lần duy nhất**

Đây là cell nặng. Sau khi `llm` đã tồn tại, **không chạy lại cell này** khi đổi retrieval / `K` / Vanilla-vs-FMRec.

In [ ]:
# ============================================================
# 9. LOAD LLM ONCE
# ============================================================

llm = UnslothBatchLLM(
    MODEL_NAME,
    MAX_SEQ_LENGTH,
    LOAD_IN_4BIT,
)

print("✓ LLM is resident in memory/GPU.")

In [ ]:
# ============================================================
# 10. RANKING HELPERS USING IN-MEMORY dataset/candidates/llm
# ============================================================

def make_request(
    uid: int,
    candidate_row: Dict[str, Any],
    lessons_by_user: Optional[Dict[int, List[str]]] = None,
) -> Dict[str, Any]:

    # All expensive/static prompt-side work was precomputed in cell 3.
    instruction = instruction_cache.get(uid, "")
    history_items = history_cache[uid]
    candidate_ids = [int(x) for x in candidate_row["candidates"]]
    candidate_items = candidate_items_cache[uid]

    facts = []
    if lessons_by_user:
        facts = list(lessons_by_user.get(uid, []))
        if MAX_LESSON_FACTS > 0:
            facts = facts[:MAX_LESSON_FACTS]

    prompt = build_memrec_vanilla_prompt(
        user_id=uid,
        instruction=instruction,
        history_items=history_items,
        candidates=candidate_items,
        failure_lessons=facts,
    )

    return {
        "uid": uid,
        "target": int(candidate_row["target"]),
        "candidates": candidate_ids,
        "n_history_items": len(history_items),
        "facts": facts,
        "prompt": prompt,
    }


def run_rank(
    lessons_by_user: Optional[Dict[int, List[str]]],
    output_dir: str,
    resume: bool = False,
    mode_name: Optional[str] = None,
):
    """
    Reuses global in-memory:
        dataset, users, candidates, llm

    lessons_by_user:
        None or {}            -> Vanilla
        retrieved_lessons    -> FMRec
    """
    has_lessons = bool(lessons_by_user)
    mode = mode_name or (
        "FMRec"
        if has_lessons
        else "Vanilla-MemRec-aligned LLMRank"
    )

    os.makedirs(output_dir, exist_ok=True)
    result_path = os.path.join(output_dir, "per_user_results.jsonl")

    if resume:
        done, old_rows = load_resume(result_path)
    else:
        done, old_rows = set(), []
        if os.path.exists(result_path):
            os.remove(result_path)

    pending = [uid for uid in users if uid not in done]

    print(
        f"Mode={mode}; users={len(users)}; pending={len(pending)}; "
        f"candidates={N_CANDIDATES}; batch={BATCH_SIZE}"
    )

    # Snapshot counters so each run gets its own timing stats even though
    # the same LLM object is reused across multiple experiments.
    stats_before = llm.stats().copy()

    new_rows = []
    pbar = tqdm(total=len(pending), desc=f"{mode} test")

    for start in range(0, len(pending), BATCH_SIZE):
        batch_users = pending[start:start + BATCH_SIZE]

        requests = [
            make_request(
                uid,
                candidates[uid],
                lessons_by_user,
            )
            for uid in batch_users
        ]

        raw_outputs = llm.generate_batch(
            [r["prompt"] for r in requests],
            MAX_NEW_TOKENS,
        )

        parsed = [
            parse_ranking(raw, req["candidates"])
            for raw, req in zip(raw_outputs, requests)
        ]

        bad = [
            idx
            for idx, result in enumerate(parsed)
            if not result[1]
        ]

        for _ in range(MAX_RETRIES):
            if not bad:
                break

            retry_prompts = [
                requests[idx]["prompt"]
                + "\n\nThe previous response was invalid. "
                  "Return exactly one score object for every candidate item, "
                  "using the original integer item_id values."
                for idx in bad
            ]

            retry_outputs = llm.generate_batch(
                retry_prompts,
                MAX_NEW_TOKENS,
            )

            new_bad = []

            for local_idx, global_idx in enumerate(bad):
                parsed[global_idx] = parse_ranking(
                    retry_outputs[local_idx],
                    requests[global_idx]["candidates"],
                )

                if not parsed[global_idx][1]:
                    new_bad.append(global_idx)

            bad = new_bad

        batch_rows = []

        for req, parsed_result in zip(requests, parsed):
            ranking, valid, score_rows = parsed_result
            target = req["target"]

            position = (
                ranking.index(target)
                if target in ranking
                else len(ranking)
            )

            batch_rows.append({
                "user_id": req["uid"],
                "target": target,
                "candidates": req["candidates"],
                "ranking": ranking,
                "rank_position": position,
                "valid_output": valid,
                "n_history_items": req["n_history_items"],
                "n_memory_facts": len(req["facts"]),
                "memory_facts": req["facts"],
                "scores": score_rows,
            })

        append_jsonl(result_path, batch_rows)
        new_rows.extend(batch_rows)

        pbar.update(len(batch_users))
        pbar.set_postfix({
            "calls": llm.calls,
            "user/s": f"{llm.stats()['examples_per_second']:.2f}",
        })

    pbar.close()

    rows = old_rows + new_rows
    user_set = set(users)
    rows = [
        row
        for row in rows
        if int(row["user_id"]) in user_set
    ]
    positions = [int(row["rank_position"]) for row in rows]

    stats_after = llm.stats()
    run_examples = stats_after["examples"] - stats_before["examples"]
    run_seconds = stats_after["seconds"] - stats_before["seconds"]
    run_stats = {
        "generate_calls": stats_after["generate_calls"] - stats_before["generate_calls"],
        "examples": run_examples,
        "seconds": run_seconds,
        "examples_per_second": (
            run_examples / run_seconds if run_seconds else 0.0
        ),
    }

    summary = {
        "mode": mode,
        "model": MODEL_NAME,
        "n_users": len(rows),
        "n_candidates": N_CANDIDATES,
        "candidate_source": "memrec_test",
        "warmup": False,
        "reflection": False,
        "use_instruction": USE_INSTRUCTION,
        "raw_history_in_final_prompt": True,
        "use_description": USE_DESCRIPTION,
        "max_lesson_facts": MAX_LESSON_FACTS if has_lessons else 0,
        "memory_coverage": (
            sum(row.get("n_memory_facts", 0) > 0 for row in rows) / len(rows)
            if rows else 0.0
        ),
        "valid_output_rate": (
            sum(bool(row.get("valid_output")) for row in rows) / len(rows)
            if rows else 0.0
        ),
        "metrics": metrics(positions),
        "llm_stats": run_stats,
        # Also useful for checking how much work this notebook kernel has done.
        "llm_stats_cumulative": stats_after,
    }

    summary_path = os.path.join(output_dir, "summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    return summary, rows

## Chạy FMRec

Cell này sử dụng trực tiếp `retrieved_lessons` đang ở RAM — **không cần đọc lại `retrieved_lessons_by_user.jsonl`** và không reload model.

In [ ]:
# ============================================================
# 11. RUN FMREC
# ============================================================

fmrec_summary, fmrec_rows = run_rank(
    lessons_by_user=retrieved_lessons,
    output_dir=FMREC_OUTPUT_DIR,
    resume=False,   # đổi True nếu muốn tiếp tục một run bị dừng
)

## Optional: chạy Vanilla bằng **chính LLM đang nằm trong GPU**

Không cần load lại model. Vanilla và FMRec vẫn dùng cùng `users` và cùng frozen `candidates`.

In [ ]:
# ============================================================
# 12. OPTIONAL — RUN VANILLA WITHOUT RELOADING LLM
# ============================================================
# Uncomment khi cần chạy baseline Vanilla trong cùng session.

# vanilla_summary, vanilla_rows = run_rank(
#     lessons_by_user=None,
#     output_dir=VANILLA_OUTPUT_DIR,
#     resume=False,
# )

## Workflow nhanh cho sensitivity `K`

Ví dụ muốn thử `K=1,2,3,5`:

1. **Không chạy lại** dataset/candidate cell.
2. **Không chạy lại** load LightGCN/failure memory cell.
3. **Không chạy lại** load LLM cell.
4. Chỉ cập nhật `TOP_K_NEIGHBORS`, chạy lại retrieval cell, rồi gọi `run_rank(...)` với output directory khác.

Ví dụ:

```python
TOP_K_NEIGHBORS = 5

retrieved_lessons, retrieval_audit, retrieval_summary = retrieve_all(
    users, user_embeddings, failure_memory_by_user,
    top_k_neighbors=TOP_K_NEIGHBORS,
    include_self=INCLUDE_SELF,
    label_sources=LABEL_SOURCES,
)

summary_k5, rows_k5 = run_rank(
    retrieved_lessons,
    output_dir="results/llmrank_fmrec_k5",
    resume=False,
)
```

Như vậy phần tốn thời gian nhất — load Gemma/Unsloth — chỉ xảy ra một lần trong kernel.